# Meta-Feature Analysis Tutorial: Methodology Walkthrough

This notebook is a publication-oriented walkthrough of the `meta-feature-analysis` methodology.
Instead of only calling `run_analysis`, we build every stage explicitly and inspect intermediate tables.

By the end, you will have:
1. an inline config (defined here, not loaded from `configs/default.yaml`)
2. split-level and dataset-level tables with explanatory checks
3. a direct before/after comparison showing how a config change alters results

## 0) Environment and scope

- Kernel/environment: `/work/mherre/tabular-meta-feature-analysis/tabarena/.venv`
- This tutorial uses a small dataset subset for fast iteration.
- To run on the full benchmark, set `selected_datasets = None` in Section 2.

In [1]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

notebook_dir = Path.cwd()
project_dir = notebook_dir.parent
repo_root = project_dir.parent

sys.path.insert(0, str(project_dir / "src"))
sys.path.insert(0, str(repo_root / "tabarena" / "tabarena"))

from mfa.aggregation import build_analysis_table
from mfa.config import parse_config
from mfa.data.loader import load_tabarena_results
from mfa.gaps.pairwise import compute_pairwise_gaps
from mfa.metafeatures import build_metafeature_table
from mfa.pipeline import run_analysis
from mfa.stats.correlation import correlate_all
from mfa.stats.correction import apply_fdr_correction
from mfa.types import AnalysisUnit, ComparisonSpec, GroupDef
from tabarena.nips2025_utils.fetch_metadata import load_task_metadata
from tabarena.nips2025_utils.tabarena_context import TabArenaContext

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

/work/mherre/tabular-meta-feature-analysis/tabarena/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-16 12:57:25,940	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## 1) Build an initial config inline (no YAML file load)

This is intentionally defined in the notebook so every change is immediately visible and auditable.

In [ ]:
raw_config = {
    "version": 1,
    "groups": {
        "gbdt": {
            "config_types": ["CAT", "GBM", "XGB"],
            "label": "GBDT",
        },
        "nn": {
            "config_types": [
                "FASTAI",
                "NN_TORCH",
                "REALMLP_GPU",
                "TABM_GPU",
                "MNCA_GPU",
                "TABDPT_GPU",
                "TABSTAR",
            ],
            "label": "NN",
        },
    },
    "comparisons": [
        {
            "name": "nn_vs_gbdt",
            "group_a": "nn",
            "group_b": "gbdt",
            "expected_direction": "positive",
        }
    ],
    "analysis": {
        "unit": "dataset",
        "error_column": "metric_error",
        "method_variant": "tuned",
        "exclude_methods_containing": ["AutoGluon"],
    },
    "metafeatures": {
        "feature_sets": ["basic", "irregularity"],
        "irregularity_components": [
            "irreg_min_cov_eig",
            "irreg_std_skew",
            "irreg_range_skew",
            "irreg_kurtosis_std",
        ],
    },
    "statistics": {
        "correlation_method": "spearman",
        "alpha": 0.05,
        "fdr_method": "bh",
        "confidence_interval": False,
        "multivariate": False,
    },
    "cache": {
        "enabled": True,
        "directory": ".mfa_cache_tutorial",
        "stages": {
            "raw_results": True,
            "metafeatures": True,
            "gaps": True,
            "statistics": False,
        },
    },
}

config = parse_config(raw_config)

config_overview = pd.DataFrame(
    [
        {"section": "analysis.unit", "value": config.analysis.unit.value},
        {"section": "analysis.method_variant", "value": config.analysis.method_variant},
        {
            "section": "metafeatures.feature_sets",
            "value": ", ".join(config.metafeatures.feature_sets),
        },
        {
            "section": "statistics.correlation_method",
            "value": config.statistics.correlation_method.value,
        },
        {
            "section": "statistics.fdr_method",
            "value": (
                None
                if config.statistics.fdr_method is None
                else config.statistics.fdr_method.value
            ),
        },
        {"section": "cache.directory", "value": str(config.cache.directory)},
    ]
)
display(config_overview)

group_overview = pd.DataFrame(
    [
        {
            "group": group.name,
            "label": group.label,
            "config_types": ", ".join(sorted(group.config_types)),
        }
        for group in config.groups.values()
    ]
)
display(group_overview)

,section,value
0,analysis.unit,dataset
1,analysis.method_variant,tuned
2,metafeatures.feature_sets,"basic, irregularity"
3,statistics.correlation_method,spearman
4,statistics.fdr_method,bh
5,cache.directory,.mfa_cache_tutorial


,group,label,config_types
0,gbdt,GBDT,"CAT, GBM, XGB"
1,nn,NN,"FASTAI, MNCA_GPU, NN_TORCH, REALMLP_GPU, TABDP..."


## 2) Choose datasets and inspect task metadata

For walkthrough notebooks, using a small subset keeps iteration fast. The code path is the same as for full runs.

In [ ]:
metadata = (
    load_task_metadata().sort_values("n_samples_train_per_fold").reset_index(drop=True)
)

selected_datasets = metadata.head(8)["name"].tolist()
# selected_datasets = None  # <- uncomment to run all benchmark datasets

meta_cols = [
    col
    for col in ["name", "tid", "n_repeats", "n_folds", "n_samples_train_per_fold"]
    if col in metadata.columns
]
display(metadata[meta_cols].head(12))

print(
    "Selected datasets:", selected_datasets if selected_datasets is not None else "ALL"
)

,name,tid,n_repeats,n_folds,n_samples_train_per_fold
0,blood-transfusion-service-center,363621,10,3,499
1,diabetes,363629,10,3,512
2,anneal,363614,10,3,599
3,QSAR_fish_toxicity,363698,10,3,605
4,credit-g,363626,10,3,667
5,maternal_health_risk,363685,10,3,676
6,concrete_compressive_strength,363625,10,3,687
7,qsar-biodeg,363696,10,3,703
8,healthcare_insurance_expenses,363675,10,3,892
9,website_phishing,363707,10,3,902


Selected datasets: ['blood-transfusion-service-center', 'diabetes', 'anneal', 'QSAR_fish_toxicity', 'credit-g', 'maternal_health_risk', 'concrete_compressive_strength', 'qsar-biodeg']


## 3) Stage A — Load and filter raw TabArena results

At this stage we only ingest result rows and apply analysis-level filters:
- `method_subtype == analysis.method_variant`
- `dataset in selected_datasets` (if provided)
- `method` exclusion patterns (e.g., names containing `AutoGluon`)

In [4]:
ta_context = TabArenaContext()

raw_results = load_tabarena_results(
    config,
    datasets=selected_datasets,
    tabarena_context=ta_context,
)

print("Raw result table shape:", raw_results.shape)
display(raw_results.head())

subtype_counts = (
    raw_results["method_subtype"]
    .value_counts(dropna=False)
    .rename_axis("method_subtype")
    .reset_index(name="n_rows")
)
display(subtype_counts)

config_type_counts = (
    raw_results["config_type"]
    .value_counts()
    .rename_axis("config_type")
    .reset_index(name="n_rows")
    .sort_values("n_rows", ascending=False)
)
display(config_type_counts.head(12))

Raw result table shape: (4560, 7)


,dataset,fold,method,metric_error,metric_error_val,config_type,method_subtype
0,QSAR_fish_toxicity,0,CAT (tuned),0.917050,0.827959,CAT,tuned
1,QSAR_fish_toxicity,1,CAT (tuned),0.796873,0.868949,CAT,tuned
2,QSAR_fish_toxicity,2,CAT (tuned),0.909045,0.841043,CAT,tuned
3,QSAR_fish_toxicity,3,CAT (tuned),0.839846,0.863845,CAT,tuned
4,QSAR_fish_toxicity,4,CAT (tuned),0.869082,0.875444,CAT,tuned


,method_subtype,n_rows
0,tuned,4560


,config_type,n_rows
0,CAT,240
10,XT,240
17,GBM,240
16,FASTAI,240
15,NN_TORCH,240
14,RF,240
13,XGB,240
12,MNCA_GPU,240
11,TABM_GPU,240
9,TABPFNV2_GPU,240


## 4) Stage B — Compute split-level pairwise gaps

For each `(dataset, repeat, fold)` and each group:
1. pick the best method in the group (lowest normalized error)
2. compute
   - `delta_raw = best_a_error - best_b_error`
   - `delta_norm = best_a_norm_error - best_b_norm_error`

Interpretation example:
- if `best_nn_norm_error = 0.62` and `best_gbdt_norm_error = 0.40`,
  then `delta_norm = 0.22` (NN is worse on that split).

In [ ]:
toy_results = pd.DataFrame(
    [
        {
            "dataset": "toy_ds",
            "fold": 0,
            "method": "NN_A",
            "metric_error": 0.22,
            "metric_error_val": 0.22,
            "config_type": "NN_A",
            "method_subtype": "tuned",
        },
        {
            "dataset": "toy_ds",
            "fold": 0,
            "method": "NN_B",
            "metric_error": 0.25,
            "metric_error_val": 0.25,
            "config_type": "NN_B",
            "method_subtype": "tuned",
        },
        {
            "dataset": "toy_ds",
            "fold": 0,
            "method": "GBDT_A",
            "metric_error": 0.20,
            "metric_error_val": 0.20,
            "config_type": "GBDT_A",
            "method_subtype": "tuned",
        },
        {
            "dataset": "toy_ds",
            "fold": 0,
            "method": "GBDT_B",
            "metric_error": 0.21,
            "metric_error_val": 0.21,
            "config_type": "GBDT_B",
            "method_subtype": "tuned",
        },
    ]
)

toy_comparison = ComparisonSpec(
    name="nn_vs_gbdt_toy",
    group_a=GroupDef(name="nn", label="NN", config_types=frozenset({"NN_A", "NN_B"})),
    group_b=GroupDef(
        name="gbdt", label="GBDT", config_types=frozenset({"GBDT_A", "GBDT_B"})
    ),
    expected_direction="positive",
)

display(toy_results.sort_values("metric_error"))

toy_gap = compute_pairwise_gaps(
    toy_results, [toy_comparison], error_column="metric_error"
)
display(
    toy_gap[
        [
            "dataset",
            "repeat",
            "fold",
            "best_a_method",
            "best_b_method",
            "best_a_error",
            "best_b_error",
            "delta_raw",
            "delta_norm",
        ]
    ]
)

gap_table = compute_pairwise_gaps(
    raw_results,
    config.comparisons,
    error_column=config.analysis.error_column,
)

print("Gap table shape:", gap_table.shape)
display(gap_table.head())

gap_summary = (
    gap_table.groupby("dataset", as_index=False)
    .agg(
        n_splits=("delta_norm", "size"),
        delta_norm_mean=("delta_norm", "mean"),
        delta_norm_std=("delta_norm", "std"),
    )
    .sort_values("dataset")
)
display(gap_summary.head(12))

,dataset,fold,method,metric_error,metric_error_val,config_type,method_subtype
2,toy_ds,0,GBDT_A,0.20,0.20,GBDT_A,tuned
3,toy_ds,0,GBDT_B,0.21,0.21,GBDT_B,tuned
0,toy_ds,0,NN_A,0.22,0.22,NN_A,tuned
1,toy_ds,0,NN_B,0.25,0.25,NN_B,tuned


,dataset,repeat,fold,best_a_method,best_b_method,best_a_error,best_b_error,delta_raw,delta_norm
0,toy_ds,0,0,NN_A,GBDT_A,0.22,0.2,0.02,1.0


Gap table shape: (240, 17)


,dataset,repeat,fold,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,best_a_method,best_a_error,best_a_norm_error,best_b_method,best_b_error,best_b_norm_error,delta_raw,delta_norm
0,QSAR_fish_toxicity,0,0,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,TABSTAR (tuned),0.896411,0.000000,XGB (tuned),0.913628,0.728143,-0.017217,-0.728143
1,QSAR_fish_toxicity,0,1,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,TABDPT_GPU (tuned),0.759013,0.000000,XGB (tuned),0.790702,0.911509,-0.031689,-0.911509
2,QSAR_fish_toxicity,0,2,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,TABSTAR (tuned),0.880724,0.000000,CAT (tuned),0.909045,0.754716,-0.028321,-0.754716
3,QSAR_fish_toxicity,1,0,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,TABSTAR (tuned),0.799694,0.000000,GBM (tuned),0.838897,0.970033,-0.039202,-0.970033
4,QSAR_fish_toxicity,1,1,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,REALMLP_GPU (tuned),0.856003,0.035664,CAT (tuned),0.869082,0.589038,-0.013079,-0.553374


,dataset,n_splits,delta_norm_mean,delta_norm_std
0,QSAR_fish_toxicity,30,-0.616783,0.323175
1,anneal,30,-0.212325,0.362986
2,blood-transfusion-service-center,30,-0.667536,0.320919
3,concrete_compressive_strength,30,-0.201095,0.276098
4,credit-g,30,-0.431752,0.328876
5,diabetes,30,-0.883230,0.126155
6,maternal_health_risk,30,-0.701554,0.260741
7,qsar-biodeg,30,-0.504111,0.417362


## 5) Stage C — Compute split-level meta-features

Meta-features are computed on **training splits only**. In this config we extract:
- basic size/composition features (`n`, `d`, `log_n`, `n_over_d`, ...)
- irregularity components + aggregated irregularity proxy

In [6]:
metafeature_table = build_metafeature_table(
    metadata,
    datasets=selected_datasets,
    feature_sets=config.metafeatures.feature_sets,
    cache_dir=config.cache.directory,
    use_cache=config.cache.enabled and config.cache.stages.metafeatures,
    pymfe_groups=config.metafeatures.pymfe_groups,
    pymfe_summary=config.metafeatures.pymfe_summary,
    irregularity_components=config.metafeatures.irregularity_components,
    cache_version=config.version,
)

print("Metafeature table shape:", metafeature_table.shape)
display(metafeature_table.head())

feature_preview_cols = [
    col
    for col in [
        "dataset",
        "repeat",
        "fold",
        "n",
        "d",
        "log_n",
        "n_over_d",
        "cat_fraction",
        "missing_fraction",
        "irregularity",
    ]
    if col in metafeature_table.columns
]
display(metafeature_table[feature_preview_cols].head(12))

Metafeature table shape: (240, 14)


,dataset,repeat,fold,n,d,log_n,n_over_d,cat_fraction,missing_fraction,irreg_min_cov_eig,irreg_std_skew,irreg_range_skew,irreg_kurtosis_std,irregularity
0,QSAR_fish_toxicity,0,0,604,6,2.781037,100.666667,0.0,0.0,0.203223,1.298274,1.216266,5.494218,-0.167638
1,QSAR_fish_toxicity,0,1,605,6,2.781755,100.833333,0.0,0.0,0.205674,1.300849,0.434721,5.672549,-0.313193
2,QSAR_fish_toxicity,0,2,605,6,2.781755,100.833333,0.0,0.0,0.201414,1.375207,0.933257,5.525627,-0.202539
3,QSAR_fish_toxicity,1,0,604,6,2.781037,100.666667,0.0,0.0,0.196434,1.213459,0.932627,5.611706,-0.227350
4,QSAR_fish_toxicity,1,1,605,6,2.781755,100.833333,0.0,0.0,0.201081,1.461885,0.933257,5.934189,-0.181461


,dataset,repeat,fold,n,d,log_n,n_over_d,cat_fraction,missing_fraction,irregularity
0,QSAR_fish_toxicity,0,0,604,6,2.781037,100.666667,0.0,0.0,-0.167638
1,QSAR_fish_toxicity,0,1,605,6,2.781755,100.833333,0.0,0.0,-0.313193
2,QSAR_fish_toxicity,0,2,605,6,2.781755,100.833333,0.0,0.0,-0.202539
3,QSAR_fish_toxicity,1,0,604,6,2.781037,100.666667,0.0,0.0,-0.227350
4,QSAR_fish_toxicity,1,1,605,6,2.781755,100.833333,0.0,0.0,-0.181461
5,QSAR_fish_toxicity,1,2,605,6,2.781755,100.833333,0.0,0.0,-0.295573
6,QSAR_fish_toxicity,2,0,604,6,2.781037,100.666667,0.0,0.0,-0.235639
7,QSAR_fish_toxicity,2,1,605,6,2.781755,100.833333,0.0,0.0,-0.205368
8,QSAR_fish_toxicity,2,2,605,6,2.781755,100.833333,0.0,0.0,-0.187645
9,QSAR_fish_toxicity,3,0,604,6,2.781037,100.666667,0.0,0.0,-0.340344


## 6) Stage D — Join gaps + meta-features, then aggregate to dataset level

We create two views:
- **fold-level** (for diagnostics, but folds are not independent)
- **dataset-level** (the inferential unit used in confirmatory analyses)

In [ ]:
analysis_table = build_analysis_table(
    gap_table,
    metafeature_table,
    unit=config.analysis.unit,
)

analysis_fold = build_analysis_table(
    gap_table,
    metafeature_table,
    unit=AnalysisUnit.FOLD,
)

print("Fold-level rows:", len(analysis_fold))
print("Dataset-level rows:", len(analysis_table))
display(analysis_table.head())

analysis_view_cols = [
    col
    for col in [
        "dataset",
        "comparison_name",
        "n_splits",
        "delta_norm",
        "delta_norm_std",
        "delta_norm_sem",
        "log_n",
        "n_over_d",
        "irregularity",
    ]
    if col in analysis_table.columns
]
display(
    analysis_table[analysis_view_cols]
    .sort_values(["comparison_name", "dataset"])
    .reset_index(drop=True)
)

Fold-level rows: 240
Dataset-level rows: 8


/tmp/ipykernel_1318577/2066351417.py:7: UserWarning: Fold-level analysis treats non-independent folds as separate observations.
  analysis_fold = build_analysis_table(


,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,log_n,n_over_d,cat_fraction,missing_fraction,irreg_min_cov_eig,irreg_std_skew,irreg_range_skew,irreg_kurtosis_std,irregularity,best_a_error,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem
0,QSAR_fish_toxicity,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,604.666667,6.0,2.781516,100.777778,0.000000,0.0,2.038879e-01,1.322848,0.875680,5.710032,-0.224730,0.848883,0.078433,0.873111,0.695217,-0.024228,0.017388,0.003175,-0.616783,0.323175,0.059003
1,anneal,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,598.666667,38.0,2.777185,15.754386,0.842105,0.0,6.344699e-01,2.233031,1.992112,6.607939,-0.356730,0.015640,0.253702,0.020887,0.466027,-0.005247,0.010249,0.001871,-0.212325,0.362986,0.066272
2,blood-transfusion-service-center,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,498.666667,4.0,2.697810,124.666667,0.000000,0.0,-3.700743e-17,1.999213,1.999924,8.026330,0.385109,0.237343,0.075377,0.253554,0.742914,-0.016211,0.010217,0.001865,-0.667536,0.320919,0.058592
3,concrete_compressive_strength,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,686.666667,8.0,2.836746,85.833333,0.000000,0.0,2.992797e-02,-0.836175,-0.831604,4.467260,-0.761568,4.059305,0.184679,4.156597,0.385774,-0.097292,0.140347,0.025624,-0.201095,0.276098,0.050408
4,credit-g,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,666.666667,20.0,2.823909,33.333333,0.650000,0.0,2.816875e-01,2.645621,2.645654,1.947403,0.231619,0.189823,0.060070,0.201499,0.491822,-0.011675,0.010225,0.001867,-0.431752,0.328876,0.060044


,dataset,comparison_name,n_splits,delta_norm,delta_norm_std,delta_norm_sem,log_n,n_over_d,irregularity
0,QSAR_fish_toxicity,nn_vs_gbdt,30,-0.616783,0.323175,0.059003,2.781516,100.777778,-0.224730
1,anneal,nn_vs_gbdt,30,-0.212325,0.362986,0.066272,2.777185,15.754386,-0.356730
2,blood-transfusion-service-center,nn_vs_gbdt,30,-0.667536,0.320919,0.058592,2.697810,124.666667,0.385109
3,concrete_compressive_strength,nn_vs_gbdt,30,-0.201095,0.276098,0.050408,2.836746,85.833333,-0.761568
4,credit-g,nn_vs_gbdt,30,-0.431752,0.328876,0.060044,2.823909,33.333333,0.231619
5,diabetes,nn_vs_gbdt,30,-0.883230,0.126155,0.023033,2.709270,64.000000,0.046498
6,maternal_health_risk,nn_vs_gbdt,30,-0.701554,0.260741,0.047605,2.829947,112.666667,-0.738189
7,qsar-biodeg,nn_vs_gbdt,30,-0.504111,0.417362,0.076200,2.846749,17.138211,1.417991


## 7) Stage E — Correlation tests + multiple-testing correction

We run hypothesis tests on selected predictors against `delta_norm`.

Because `expected_direction` is set to `positive`, the package reports **one-sided** p-values aligned to that direction.

In [ ]:
predictors = ["log_n", "n_over_d", "irregularity"]

correlation_results = correlate_all(
    analysis_table,
    comparisons=config.comparisons,
    predictors=predictors,
    target="delta_norm",
    method=config.statistics.correlation_method,
    confidence_interval=config.statistics.confidence_interval,
    ci_bootstrap_samples=config.statistics.ci_bootstrap_samples,
    ci_confidence_level=config.statistics.ci_confidence_level,
)

corr_df = pd.DataFrame([result.__dict__ for result in correlation_results]).sort_values(
    "p_value"
)
display(corr_df)

correction_result = apply_fdr_correction(
    correlation_results,
    method=config.statistics.fdr_method,
    alpha=config.statistics.alpha,
)

if correction_result is not None:
    corr_df = corr_df.copy()
    adj_map = {
        (result.predictor, result.comparison_name): p_adj
        for result, p_adj in zip(
            correction_result.results, correction_result.adjusted_p_values
        )
    }
    reject_map = {
        (result.predictor, result.comparison_name): rejected
        for result, rejected in zip(
            correction_result.results, correction_result.rejected
        )
    }
    corr_df["p_value_adj"] = [
        adj_map[(row.predictor, row.comparison_name)]
        for row in corr_df.itertuples(index=False)
    ]
    corr_df["rejected"] = [
        reject_map[(row.predictor, row.comparison_name)]
        for row in corr_df.itertuples(index=False)
    ]

display(corr_df.sort_values("p_value"))

,comparison_name,predictor,target,statistic,p_value,n_observations,ci_lower,ci_upper,direction_confirmed
0,nn_vs_gbdt,log_n,delta_norm,0.428571,0.144702,8,None,None,True
2,nn_vs_gbdt,irregularity,delta_norm,-0.261905,0.734539,8,None,None,False
1,nn_vs_gbdt,n_over_d,delta_norm,-0.500000,0.896484,8,None,None,False


,comparison_name,predictor,target,statistic,p_value,n_observations,ci_lower,ci_upper,direction_confirmed,p_value_adj,rejected
0,nn_vs_gbdt,log_n,delta_norm,0.428571,0.144702,8,None,None,True,0.434105,False
2,nn_vs_gbdt,irregularity,delta_norm,-0.261905,0.734539,8,None,None,False,0.896484,False
1,nn_vs_gbdt,n_over_d,delta_norm,-0.500000,0.896484,8,None,None,False,0.896484,False


## 8) Observe a config change directly

We now change one knob only:
- `analysis.method_variant: tuned -> tuned_ensemble`

Then we recompute and compare tables side-by-side.

In [ ]:
def run_manual_pipeline(parsed_config):
    local_raw = load_tabarena_results(
        parsed_config,
        datasets=selected_datasets,
        tabarena_context=ta_context,
    )
    local_gaps = compute_pairwise_gaps(
        local_raw,
        parsed_config.comparisons,
        error_column=parsed_config.analysis.error_column,
    )
    local_analysis = build_analysis_table(
        local_gaps,
        metafeature_table,
        unit=parsed_config.analysis.unit,
    )
    local_corr = correlate_all(
        local_analysis,
        comparisons=parsed_config.comparisons,
        predictors=predictors,
        target="delta_norm",
        method=parsed_config.statistics.correlation_method,
        confidence_interval=False,
    )
    local_corr_df = pd.DataFrame([result.__dict__ for result in local_corr])[
        ["comparison_name", "predictor", "statistic", "p_value"]
    ]
    return local_raw, local_gaps, local_analysis, local_corr_df


base_raw, base_gaps, base_analysis, base_corr_df = run_manual_pipeline(config)

variant_config_dict = copy.deepcopy(raw_config)
variant_config_dict["analysis"]["method_variant"] = "tuned_ensemble"
variant_config = parse_config(variant_config_dict)

variant_raw, variant_gaps, variant_analysis, variant_corr_df = run_manual_pipeline(
    variant_config
)

print("Rows after subtype filter:")
display(
    pd.DataFrame(
        [
            {
                "scenario": "tuned (base)",
                "raw_rows": len(base_raw),
                "gap_rows": len(base_gaps),
                "analysis_rows": len(base_analysis),
            },
            {
                "scenario": "tuned_ensemble (variant)",
                "raw_rows": len(variant_raw),
                "gap_rows": len(variant_gaps),
                "analysis_rows": len(variant_analysis),
            },
        ]
    )
)

delta_compare = base_analysis[["dataset", "delta_norm"]].merge(
    variant_analysis[["dataset", "delta_norm"]],
    on="dataset",
    suffixes=("_tuned", "_tuned_ensemble"),
)
delta_compare["delta_shift"] = (
    delta_compare["delta_norm_tuned_ensemble"] - delta_compare["delta_norm_tuned"]
)
delta_compare["abs_shift"] = delta_compare["delta_shift"].abs()
display(delta_compare.sort_values("abs_shift", ascending=False).reset_index(drop=True))

corr_compare = base_corr_df.merge(
    variant_corr_df,
    on=["comparison_name", "predictor"],
    suffixes=("_tuned", "_tuned_ensemble"),
)
display(corr_compare.sort_values("predictor").reset_index(drop=True))

Rows after subtype filter:


,scenario,raw_rows,gap_rows,analysis_rows
0,tuned (base),4560,240,8
1,tuned_ensemble (variant),4560,240,8


,dataset,delta_norm_tuned,delta_norm_tuned_ensemble,delta_shift,abs_shift
0,concrete_compressive_strength,-0.201095,-0.350273,-0.149178,0.149178
1,qsar-biodeg,-0.504111,-0.651246,-0.147135,0.147135
2,maternal_health_risk,-0.701554,-0.846624,-0.145071,0.145071
3,QSAR_fish_toxicity,-0.616783,-0.759974,-0.143190,0.143190
4,anneal,-0.212325,-0.299178,-0.086853,0.086853
5,credit-g,-0.431752,-0.499560,-0.067808,0.067808
6,blood-transfusion-service-center,-0.667536,-0.601675,0.065862,0.065862
7,diabetes,-0.883230,-0.885996,-0.002766,0.002766


,comparison_name,predictor,statistic_tuned,p_value_tuned,statistic_tuned_ensemble,p_value_tuned_ensemble
0,nn_vs_gbdt,irregularity,-0.261905,0.734539,-0.166667,0.653381
1,nn_vs_gbdt,log_n,0.428571,0.144702,0.095238,0.411253
2,nn_vs_gbdt,n_over_d,-0.500000,0.896484,-0.404762,0.840056


## 9) Sanity check: manual walkthrough vs `run_analysis`

The orchestrated API should match the manually built table for the same config.

In [ ]:
result = run_analysis(
    config,
    datasets=selected_datasets,
    task_metadata=metadata,
    tabarena_context=ta_context,
)

manual_sorted = analysis_table.sort_values(["comparison_name", "dataset"]).reset_index(
    drop=True
)
orchestrated_sorted = result.analysis_table.sort_values(
    ["comparison_name", "dataset"]
).reset_index(drop=True)

print(
    "Manual stage-by-stage table equals run_analysis output:",
    manual_sorted.equals(orchestrated_sorted),
)
display(orchestrated_sorted.head())

Manual stage-by-stage table equals run_analysis output: True


/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/src/mfa/stats/correlation.py:31: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  statistic, p_value = spearmanr(x, y)


,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,log_n,n_over_d,cat_fraction,missing_fraction,irreg_min_cov_eig,irreg_std_skew,irreg_range_skew,irreg_kurtosis_std,irregularity,best_a_error,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem
0,QSAR_fish_toxicity,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,604.666667,6.0,2.781516,100.777778,0.000000,0.0,2.038879e-01,1.322848,0.875680,5.710032,-0.224730,0.848883,0.078433,0.873111,0.695217,-0.024228,0.017388,0.003175,-0.616783,0.323175,0.059003
1,anneal,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,598.666667,38.0,2.777185,15.754386,0.842105,0.0,6.344699e-01,2.233031,1.992112,6.607939,-0.356730,0.015640,0.253702,0.020887,0.466027,-0.005247,0.010249,0.001871,-0.212325,0.362986,0.066272
2,blood-transfusion-service-center,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,498.666667,4.0,2.697810,124.666667,0.000000,0.0,-3.700743e-17,1.999213,1.999924,8.026330,0.385109,0.237343,0.075377,0.253554,0.742914,-0.016211,0.010217,0.001865,-0.667536,0.320919,0.058592
3,concrete_compressive_strength,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,686.666667,8.0,2.836746,85.833333,0.000000,0.0,2.992797e-02,-0.836175,-0.831604,4.467260,-0.761568,4.059305,0.184679,4.156597,0.385774,-0.097292,0.140347,0.025624,-0.201095,0.276098,0.050408
4,credit-g,nn_vs_gbdt,nn,gbdt,NN,GBDT,positive,30,666.666667,20.0,2.823909,33.333333,0.650000,0.0,2.816875e-01,2.645621,2.645654,1.947403,0.231619,0.189823,0.060070,0.201499,0.491822,-0.011675,0.010225,0.001867,-0.431752,0.328876,0.060044


## 10) What to vary for publication sensitivity checks

Typical high-impact knobs:
- `analysis.method_variant` (`default`, `tuned`, `tuned_ensemble`)
- `groups.*.config_types` (family definitions)
- `metafeatures.feature_sets` and optional `pymfe` settings
- `statistics.confidence_interval = True` for bootstrap CIs in final reporting